<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/ols_plus3_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install wrds

In [ ]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt

In [ ]:
# author's datashare
import os
import pandas as pd

csv_path = "/content/drive/MyDrive/datashare.csv"
df = pd.read_csv(csv_path)
df["DATE"] = pd.to_datetime(df["DATE"], format="%Y%m%d")

In [ ]:
db = wrds.Connection()

Enter your WRDS username [root]:zixian_zhou
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [ ]:
permnos = df['permno'].unique().tolist()
dates = df['DATE'].unique().tolist()
start_date = min(dates)
end_date = max(dates)


query = f"""
SELECT
    b.permno,
    b.date,
    b.ret,
    c.rf,
    (b.ret - c.rf) AS exret
FROM crsp.msf b
LEFT JOIN ff.factors_monthly c ON
    EXTRACT(YEAR FROM b.date) = EXTRACT(YEAR FROM c.date)
    AND EXTRACT(MONTH FROM b.date) = EXTRACT(MONTH FROM c.date)
WHERE b.date >= '{start_date}' AND b.date <= '{end_date}'
  AND b.ret IS NOT NULL
"""

df_msf_all = db.raw_sql(query)
df['DATE'] = pd.to_datetime(df['DATE'])
df_msf_all['date'] = pd.to_datetime(df_msf_all['date'])

df_exret = pd.merge(
    df,
    df_msf_all,
    left_on=['permno', 'DATE'],
    right_on=['permno', 'date'],
    how='inner'
)

In [ ]:
df_exret = df_exret.drop(columns=['date'])

In [ ]:
!pip -q install gspread google-api-python-client

from google.colab import auth
auth.authenticate_user()

In [ ]:
# 8 macro variables from websites —— FIXED for Google Sheet in Drive (Data2024)

import os
import pandas as pd
from google.colab import drive

# 确保 Drive 已挂载（你前面已经 mount 了，这里再写一次也不影响）
drive.mount('/content/drive', force_remount=False)

# 1) 在 MyDrive 里找到名为 "Data2024" 的 Google Sheet，并导出成 xlsx
#    你图里是绿色 Sheets 图标 => 它是 Google Sheet，不是 .xlsx 文件
import gspread
from google.auth import default
from googleapiclient.discovery import build

# 授权
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_NAME = "Data2024"   # 就是你图里的名字

# 打开 Google Sheet
sh = gc.open(SHEET_NAME)

# 用 Drive API 导出为 xlsx
drive_service = build('drive', 'v3', credentials=creds)

file_id = sh.id  # Google Sheet 的 file id
export_path = "/content/Data2024_export.xlsx"

request = drive_service.files().export_media(
    fileId=file_id,
    mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
)

with open(export_path, "wb") as f:
    f.write(request.execute())

print("✅ Exported Google Sheet to:", export_path)

# 2) 读取 Monthly sheet（如果你的 sheet 名不是 Monthly，把这里改成实际名字）
df_macro = pd.read_excel(export_path, sheet_name="Monthly")

macro = ['tbl','d/p','e/p','b/m','tms','dfy','ntis','svar']

# 3) 排序 + shift（避免 look-ahead bias）
df_macro = df_macro.sort_values('yyyymm').reset_index(drop=True)
df_macro[macro] = df_macro[macro].shift(1)

print("✅ df_macro shape:", df_macro.shape)
df_macro.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Exported Google Sheet to: /content/Data2024_export.xlsx
✅ df_macro shape: (1848, 57)


,yyyymm,price,d12,e12,ret,retx,AAA,BAA,lty,ltr,...,ygap,rdsp,rsvix,gpce,gip,tchi,house,avgcor,shtint,disag
0,187101,4.44,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,187102,4.50,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,187103,4.61,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,187104,4.74,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,187105,4.86,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_macro['date'] = pd.to_datetime(df_macro['yyyymm'], format='%Y%m')
df_macro['year_month'] = df_macro['date'].dt.to_period('M')


In [ ]:
df_macro=df_macro[['yyyymm','tbl','d/p','e/p','b/m','tms','dfy','ntis','svar','year_month']]
# df_macro

In [ ]:
df_exret['year_month'] = df_exret['DATE'].dt.to_period('M')
df_exret = df_exret[df_exret['ret'].notna()]
df_exret = df_exret.reset_index(drop=True)
# df_exret

In [ ]:
# --- Make target = exret_{t+1} ---
df_exret = df_exret.sort_values(['permno', 'DATE']).reset_index(drop=True)
df_exret['exret_lead1'] = df_exret.groupby('permno')['exret'].shift(-1)

# Paper sample window (optional but recommended for matching Table results)
df_exret = df_exret[(df_exret['DATE'] >= pd.Timestamp('1957-03-31')) &
                    (df_exret['DATE'] <= pd.Timestamp('2016-12-31'))].copy()

df_exret = df_exret.dropna(subset=['exret_lead1']).reset_index(drop=True)

In [ ]:
# ===== OLS-3 predictors only =====
CHAR_COLS = ['mvel1', 'bm', 'mom12m']
features = CHAR_COLS  # 只处理这三列，避免误伤其它列（尤其是 target）

def rank_norm_median(s: pd.Series) -> pd.Series:
    # 1) 当月截面中位数填缺失
    med = s.median(skipna=True)
    s = s.fillna(med)

    # 2) rank 映射到 [-1, 1]
    r = s.rank(method='average')
    n = r.count()
    return (r / (n + 1)) * 2 - 1

# 用 transform 比 apply 更稳，不会乱 index，也更快
df_exret[features] = df_exret.groupby('DATE')[features].transform(rank_norm_median)

df_exret[features].describe().T
# features

,count,mean,std,min,25%,50%,75%,max
mvel1,3715725.0,-5.935651e-18,0.577239,-0.999777,-0.499912,0.0,0.499912,0.999777
bm,3647031.0,2.470415e-18,0.567101,-0.999022,-0.486858,0.0,0.486862,0.999022
mom12m,3715725.0,-4.910680e-18,0.576178,-0.995516,-0.499319,0.0,0.499322,0.995589


In [ ]:
# final data
df_merged = pd.merge(df_exret, df_macro, on='year_month', how='left')
df_merged = df_merged.drop(columns=['yyyymm','year_month'])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, HuberRegressor

# ===== 0) 核心设定（OLS-3 + Huber） =====
CHAR_COLS = ['mvel1', 'bm', 'mom12m']
TARGET = 'exret_lead1'   # 你前面已经生成了 exret_{t+1}

VALIDATION_END  = pd.Timestamp('1986-12-31')
TEST_END        = pd.Timestamp('2016-12-31')
VALIDATION_YEARS = 12

# ===== 1) 选择用于回归的数据（注意：REQUIRED 不要包含 MACRO_COLS） =====
# 你可以用 df_exret 或 df_merged；这里用 df_merged（包含 mvel1 也方便后面 Top/Bottom1000）
df = df_merged.copy()
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['DATE', 'permno']).reset_index(drop=True)

REQUIRED = CHAR_COLS + [TARGET]
df_clean = df.dropna(subset=REQUIRED).copy()

# ===== 2) feature matrix（OLS-3 只有 3 列） =====
def build_feature_matrix(data):
    return data[CHAR_COLS].to_numpy(dtype=np.float64)

# ===== 3) OOS R2（baseline=0） =====
def oos_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum(y_true ** 2)
    return 1.0 - ss_res / ss_tot

# ===== 4) 测试年份 =====
test_years = sorted(
    df_clean.loc[
        (df_clean['DATE'] > VALIDATION_END) & (df_clean['DATE'] <= TEST_END),
        'DATE'
    ].dt.year.unique()
)

print(f"Clean rows: {len(df_clean):,}")
print(f"Test years: {test_years[0]}–{test_years[-1]} ({len(test_years)} years)")

# ===== 5) Rolling: expanding train + rolling 12y validation，验证集定 xi(99.9%) =====
all_preds = []

for year in test_years:
    train_cutoff = pd.Timestamp(f'{year - 1}-12-31')
    val_start    = pd.Timestamp(f'{year - 1 - VALIDATION_YEARS}-12-31')

    df_train = df_clean[df_clean['DATE'] <= val_start]
    df_val   = df_clean[(df_clean['DATE'] > val_start) & (df_clean['DATE'] <= train_cutoff)]
    df_test  = df_clean[df_clean['DATE'].dt.year == year]

    if len(df_train) == 0 or len(df_val) == 0 or len(df_test) == 0:
        continue

    X_train = build_feature_matrix(df_train)
    X_val   = build_feature_matrix(df_val)
    X_test  = build_feature_matrix(df_test)

    # ✅ 关键：把 y 乘 100，避免 sklearn 要求 epsilon > 1 的硬约束
    y_train = df_train[TARGET].to_numpy(dtype=np.float64) * 100.0
    y_val   = df_val[TARGET].to_numpy(dtype=np.float64)   * 100.0
    y_test  = df_test[TARGET].to_numpy(dtype=np.float64)  * 100.0

    # Step 1: 先 OLS 拟合训练集 -> 得到验证集残差
    ols_tmp = LinearRegression(fit_intercept=True)
    ols_tmp.fit(X_train, y_train)
    val_resid = np.abs(y_val - ols_tmp.predict(X_val))

    # Step 2: xi = 99.9% quantile
    xi = float(np.quantile(val_resid, 0.999))
    xi = max(xi, 1.0001)   # sklearn epsilon 必须 > 1

    # Step 3: 用 Huber 在 train+val 上拟合
    X_fit = np.vstack([X_train, X_val])
    y_fit = np.concatenate([y_train, y_val])

    huber = HuberRegressor(epsilon=xi, fit_intercept=True, alpha=0.0, max_iter=500)
    huber.fit(X_fit, y_fit)
    y_pred = huber.predict(X_test)

    res = df_test[['DATE', 'permno', 'mvel1']].copy().reset_index(drop=True)
    res['y_true'] = y_test
    res['y_pred'] = y_pred
    all_preds.append(res)

    print(f"Year {year} | train {len(df_train):,} | val {len(df_val):,} | test {len(df_test):,} | xi={xi:.4f}")

# ===== 6) 汇总评估 =====
results = pd.concat(all_preds, ignore_index=True)

r2_all = oos_r2(results['y_true'], results['y_pred'])
print("\n" + "═"*75)
print(f"{'Subsample':<30}  {'OOS R2':>10}  {'Expected (Paper)':>20}")
print("─"*75)
print(f"{'All stocks (panel)':<30}  {r2_all*100:>+10.4f}%  {'~ +0.16%':>20}")

for label, largest, paper_val in [
    ('Top 1000 (largest mvel1)', True,  0.34),
    ('Bottom 1000 (smallest mvel1)', False, 0.05),
]:
    if largest:
        sub = results.sort_values(['DATE','mvel1'], ascending=[True, False]).groupby('DATE').head(1000)
    else:
        sub = results.sort_values(['DATE','mvel1'], ascending=[True, True]).groupby('DATE').head(1000)

    r2 = oos_r2(sub['y_true'], sub['y_pred'])
    print(f"{label:<30}  {r2*100:>+10.4f}%  {f'~ {paper_val:+.2f}%':>20}")

print("═"*75)

Clean rows: 3,647,031
Test years: 1987–2016 (30 years)
Year 1987 | train 404,314 | val 766,206 | test 82,439 | xi=119.7593
Year 1988 | train 462,708 | val 790,251 | test 83,441 | xi=121.3641
Year 1989 | train 521,381 | val 815,019 | test 81,242 | xi=123.7732
Year 1990 | train 580,355 | val 837,287 | test 80,226 | xi=125.1704
Year 1991 | train 638,024 | val 859,844 | test 79,300 | xi=131.3531
Year 1992 | train 695,153 | val 882,015 | test 80,989 | xi=142.6269
Year 1993 | train 752,783 | val 905,374 | test 86,181 | xi=145.5898
Year 1994 | train 814,346 | val 929,992 | test 95,115 | xi=148.3885
Year 1995 | train 877,930 | val 961,523 | test 97,768 | xi=143.1760
Year 1996 | train 945,944 | val 991,277 | test 103,307 | xi=143.5413
Year 1997 | train 1,020,416 | val 1,020,112 | test 107,221 | xi=144.1398
Year 1998 | train 1,094,725 | val 1,053,024 | test 105,705 | xi=143.6770
Year 1999 | train 1,170,520 | val 1,082,934 | test 99,921 | xi=148.9411
Year 2000 | train 1,252,959 | val 1,100,416 | 

In [ ]:
from sklearn.linear_model import HuberRegressor
# ── 1. 承接前情提要的数据 ──────────────────────────────────────────────────
# 直接使用你预处理好的 df_merged
df = df_merged.copy()

# 确保 DATE 是 datetime 格式并排序
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['DATE', 'permno']).reset_index(drop=True)

print(f"Data shape:     {df.shape}")
print(f"Date range:     {df['DATE'].min().date()} to {df['DATE'].max().date()}")

# ── 2. 定义 OLS-3 的核心变量 ───────────────────────────────────────────────
TARGET = 'exret'

# OLS-3 仅保留三个最经典的异象特征：市值(mvel1)、账面市值比(bm)、动量(mom12m)
CHAR_COLS = ['mvel1', 'bm', 'mom12m']

# 8个宏观变量
MACRO_COLS = ['tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar']

# 检查列是否存在
CHAR_COLS  = [c for c in CHAR_COLS  if c in df.columns]
MACRO_COLS = [c for c in MACRO_COLS if c in df.columns]

# ── 3. 剔除含有缺失值的行 ──────────────────────────────────────────────────
TARGET = 'exret'
...
REQUIRED = CHAR_COLS + MACRO_COLS + [TARGET]

df_clean = df.dropna(subset=REQUIRED).copy()
print(f"\nClean rows (no NaN): {len(df_clean):,} / {len(df):,}")

Data shape:     (3715725, 109)
Date range:     1957-04-30 to 2016-12-30

Clean rows (no NaN): 3,647,031 / 3,715,725


In [ ]:
# ── 4. 定义样本外 R2 评估函数 (Equation 19) ────────────────────────────────
def oos_r2(y_true, y_pred):
    """
    R2_oos = 1 - sum(r - r_hat)^2 / sum(r^2)
    这是一个以0为基准的 R2（而不是以历史均值为基准），这是论文核心的评估标准。
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum(y_true ** 2)
    return 1.0 - ss_res / ss_tot

def build_feature_matrix(data):
    """
    OLS-3: 只用 3 个特征 (mvel1, bm, mom12m)，不包含宏观交互项。
    X 维度 = (n_obs, 3)
    """
    X = data[CHAR_COLS].to_numpy(dtype=np.float64)
    return X

In [ ]:
from sklearn.linear_model import LinearRegression, HuberRegressor

VALIDATION_END = pd.Timestamp('1986-12-31')
TEST_END       = pd.Timestamp('2016-12-31')

test_years = sorted(
    df_clean.loc[
        (df_clean['DATE'] > VALIDATION_END) &
        (df_clean['DATE'] <= TEST_END),
        'DATE'
    ].dt.year.unique()
)

print(f"Test period: {test_years[0]}–{test_years[-1]} ({len(test_years)} years)")
print(f"OLS-3(+H) features: {CHAR_COLS} (dim={len(CHAR_COLS)})")

VALIDATION_YEARS = 12
all_preds = []

for year in test_years:
    train_cutoff = pd.Timestamp(f'{year - 1}-12-31')
    val_start    = pd.Timestamp(f'{year - 1 - VALIDATION_YEARS}-12-31')

    # Train: <= val_start (expanding)
    df_train = df_clean[df_clean['DATE'] <= val_start]
    # Val: (val_start, train_cutoff]  (rolling 12y)
    df_val   = df_clean[(df_clean['DATE'] > val_start) & (df_clean['DATE'] <= train_cutoff)]
    # Test: current year
    df_test  = df_clean[df_clean['DATE'].dt.year == year]

    if len(df_train) == 0 or len(df_val) == 0 or len(df_test) == 0:
        continue

    X_train = build_feature_matrix(df_train)
    X_val   = build_feature_matrix(df_val)
    X_test  = build_feature_matrix(df_test)

    # （可选但推荐）把收益乘100：避免 epsilon<1 的硬约束问题；R2_oos 对尺度不敏感
    y_train = (df_train[TARGET].to_numpy(dtype=np.float64) * 100.0)
    y_val   = (df_val[TARGET].to_numpy(dtype=np.float64)   * 100.0)
    y_test  = (df_test[TARGET].to_numpy(dtype=np.float64)  * 100.0)

    # Step 1: OLS on training -> validation residuals
    ols_tmp = LinearRegression(fit_intercept=True)
    ols_tmp.fit(X_train, y_train)
    val_pred  = ols_tmp.predict(X_val)
    val_resid = np.abs(y_val - val_pred)

    # Paper xi: 99.9% quantile of |resid|
    xi = np.quantile(val_resid, 0.999)

    # Convert absolute xi to sklearn epsilon approx: epsilon * scale ~= xi
    # robust scale estimate:
    scale_hat = 1.4826 * np.median(np.abs(val_resid - np.median(val_resid)))
    if not np.isfinite(scale_hat) or scale_hat <= 1e-12:
        scale_hat = np.std(val_resid) + 1e-12

    eps = xi / scale_hat
    eps = float(max(eps, 1.0001))  # sklearn requires epsilon > 1.0

    # Step 3: refit on train+val with tuned epsilon
    X_fit = np.vstack([X_train, X_val])
    y_fit = np.concatenate([y_train, y_val])

    huber = HuberRegressor(epsilon=eps, fit_intercept=True, alpha=0.0, max_iter=500)
    huber.fit(X_fit, y_fit)
    y_pred = huber.predict(X_test)

    # 保存预测（注意：这里 y_true/y_pred 都是 *100 后的；R2 不受影响）
    res = df_test[['DATE', 'permno', 'mvel1']].copy().reset_index(drop=True)
    res['y_true'] = y_test
    res['y_pred'] = y_pred
    all_preds.append(res)

    print(f"Year {year} | train {len(df_train):,} | val {len(df_val):,} | test {len(df_test):,} | xi={xi:.4f} | eps={eps:.4f}")

Test period: 1987–2016 (30 years)
OLS-3(+H) features: ['mvel1', 'bm', 'mom12m'] (dim=3)
Year 1987 | train 404,314 | val 766,206 | test 82,439 | xi=123.6603 | eps=17.8031
Year 1988 | train 462,708 | val 790,251 | test 83,441 | xi=121.8561 | eps=17.2407
Year 1989 | train 521,381 | val 815,019 | test 81,242 | xi=123.0030 | eps=17.5000
Year 1990 | train 580,355 | val 837,287 | test 80,226 | xi=124.8555 | eps=17.6633
Year 1991 | train 638,024 | val 859,844 | test 79,300 | xi=128.6227 | eps=17.8773
Year 1992 | train 695,153 | val 882,015 | test 80,989 | xi=138.2857 | eps=18.9311
Year 1993 | train 752,783 | val 905,374 | test 86,181 | xi=143.0855 | eps=19.7126
Year 1994 | train 814,346 | val 929,992 | test 95,115 | xi=145.3542 | eps=20.0760
Year 1995 | train 877,930 | val 961,523 | test 97,768 | xi=143.4129 | eps=20.1879
Year 1996 | train 945,944 | val 991,277 | test 103,307 | xi=141.2909 | eps=20.2906
Year 1997 | train 1,020,416 | val 1,020,112 | test 107,221 | xi=142.8324 | eps=20.4977
Year

In [ ]:
# ── 7. 打印评估结果（只保留 All stocks）───────────────────────────────────────
results = pd.concat(all_preds, ignore_index=True)

# 防呆：万一 all_preds 为空
if results.empty:
    print("❌ results 为空：all_preds 没有累积任何年份的预测结果。请先确认循环是否运行、df_train/df_val/df_test 是否为空。")
else:
    r2_all = oos_r2(results['y_true'], results['y_pred'])

    print("\n" + "═"*75)
    print(f"  {'Subsample':<30}  {'OOS R2':>10}  {'Expected (Paper)':>20}")
    print("─"*75)
    print(f"  {'All stocks (panel)':<30}  {r2_all*100:>+10.4f}%  {'~ +0.16%':>20}")
    print("═"*75)


═══════════════════════════════════════════════════════════════════════════
  Subsample                           OOS R2      Expected (Paper)
───────────────────────────────────────────────────────────────────────────
  All stocks (panel)                 +0.1761%              ~ +0.16%
═══════════════════════════════════════════════════════════════════════════
